## CLASS MAPPING
## Objectif

L'objectif de cette étape est de construire une correspondance officielle et stable entre les noms des classes du dataset AgriLens et des identifiants numériques uniques.

Les algorithmes de Machine Learning et de Deep Learning ne peuvent pas utiliser directement des labels textuels tels que :

Tomato___Late_blight
Tomato___healthy
pod_borer

Ils nécessitent une représentation numérique des classes afin de pouvoir apprendre, prédire et évaluer leurs performances.

Le Class Mapping permet donc d'associer chaque classe à un identifiant unique et reproductible qui sera utilisé dans l'ensemble du projet.

### Exemple :

Tomato___Late_blight → 6

Tomato___healthy → 13

pod_borer → 16

Pourquoi créer un Class Mapping ?

Les labels textuels sont faciles à interpréter pour un humain mais ne peuvent pas être exploités directement par la plupart des modèles de classification.

Lors de l'entraînement, chaque image doit être associée à une valeur numérique représentant sa catégorie.

Le Class Mapping définit cette correspondance une seule fois afin de garantir la cohérence des données tout au long du cycle de vie du projet.

Rôle dans le projet AgriLens

Le Class Mapping constitue la référence officielle des classes du projet.

Il intervient dans plusieurs étapes :

Préparation des données

Chaque image est associée à un identifiant numérique unique correspondant à sa classe.

Entraînement des modèles

Les pipelines Machine Learning et Deep Learning utilisent les identifiants numériques comme variables cibles.

### Évaluation

Les matrices de confusion, rapports de classification et métriques de performance reposent sur cette correspondance pour interpréter correctement les résultats.

Inférence

Les prédictions numériques produites par les modèles peuvent être reconverties en labels compréhensibles pour l'utilisateur.

## Exemple :

6 → Tomato___Late_blight

Livrables produits

Cette étape génère deux artefacts officiels.

class_mapping.csv

Table de référence contenant la correspondance entre :

class_id
class_name
label2id.json

Dictionnaire utilisé directement dans les pipelines d'entraînement et d'inférence.

### Exemple :

{ "Tomato___Late_blight": 6, "Tomato___healthy": 13, "pod_borer": 16 }

In [11]:
import pandas as pd

In [12]:
df = pd.read_csv("../../01_data/metadata/inventory_v3.csv")

print(df.shape)
df.head()

(26397, 20)


,source,culture,split,classe,filename,filepath,extension,format,width,height,mode,channels,problem,ratio,brightness_mean,exposure_status,md5_hash,duplicate_exact,leakage_risk,phash
0,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_1.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,102.669785,normale,3cf3901bd20e3910ce2bd2a34285e151,False,False,da95a424acb2f2d3
1,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_10.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,121.778247,normale,2473be012438c8db2a0c1f454c226928,False,False,eef242f7a310b18a
2,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_100.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,120.446264,normale,af88d5ff63102933d009e06602aa9aa8,False,False,d1f58ff12f01ba04
3,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_101.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,139.776007,normale,d33754ff3dc4621047583561bc442850,False,False,a116ee225d6a496f
4,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_102.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,125.152757,normale,d7b2a39d2eac0a869f5510581943bed2,False,False,eeec88a22f1db215


In [13]:
classes = sorted(df["classe"].unique())

print(f"Nombre de classes : {len(classes)}")

for i, c in enumerate(classes):
    print(i, c)

Nombre de classes : 17
0 Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
1 Corn_(maize)___Common_rust_
2 Corn_(maize)___Northern_Leaf_Blight
3 Corn_(maize)___healthy
4 Tomato___Bacterial_spot
5 Tomato___Early_blight
6 Tomato___Late_blight
7 Tomato___Leaf_Mold
8 Tomato___Septoria_leaf_spot
9 Tomato___Spider_mites Two-spotted_spider_mite
10 Tomato___Target_Spot
11 Tomato___Tomato_Yellow_Leaf_Curl_Virus
12 Tomato___Tomato_mosaic_virus
13 Tomato___healthy
14 black_pod_rot
15 healthy
16 pod_borer


# Construction du Class Mapping

À partir de la liste des classes uniques, nous construisons une table officielle reliant chaque nom de classe à un identifiant numérique.

Cette table constituera la référence utilisée par les futurs pipelines Machine Learning et Deep Learning.

In [14]:
mapping = pd.DataFrame({
    "class_id": range(len(classes)),
    "class_name": classes
})

mapping

,class_id,class_name
0,0,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_...
1,1,Corn_(maize)___Common_rust_
2,2,Corn_(maize)___Northern_Leaf_Blight
3,3,Corn_(maize)___healthy
4,4,Tomato___Bacterial_spot
5,5,Tomato___Early_blight
6,6,Tomato___Late_blight
7,7,Tomato___Leaf_Mold
8,8,Tomato___Septoria_leaf_spot
9,9,Tomato___Spider_mites Two-spotted_spider_mite


# Export du fichier class_mapping.csv

Cette étape enregistre le référentiel officiel des classes dans le dossier metadata.

Ce fichier sera utilisé par les futures phases d'entraînement, d'évaluation et de déploiement.

In [15]:
mapping.to_csv(
    "../../01_data/metadata/class_mapping.csv",
    index=False
)

print("class_mapping.csv créé avec succès")

class_mapping.csv créé avec succès


# Création du dictionnaire label2id

Les frameworks de Deep Learning utilisent fréquemment des dictionnaires permettant de convertir automatiquement les labels textuels en identifiants numériques.

Nous construisons donc un dictionnaire label2id à partir du Class Mapping officiel.


In [16]:
import json

label2id = {
    c: i
    for i, c in enumerate(classes)
}

label2id

{'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 0,
 'Corn_(maize)___Common_rust_': 1,
 'Corn_(maize)___Northern_Leaf_Blight': 2,
 'Corn_(maize)___healthy': 3,
 'Tomato___Bacterial_spot': 4,
 'Tomato___Early_blight': 5,
 'Tomato___Late_blight': 6,
 'Tomato___Leaf_Mold': 7,
 'Tomato___Septoria_leaf_spot': 8,
 'Tomato___Spider_mites Two-spotted_spider_mite': 9,
 'Tomato___Target_Spot': 10,
 'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 11,
 'Tomato___Tomato_mosaic_virus': 12,
 'Tomato___healthy': 13,
 'black_pod_rot': 14,
 'healthy': 15,
 'pod_borer': 16}

# Export du fichier label2id.json

Cette étape sauvegarde le dictionnaire label2id dans le dossier metadata.

Ce fichier pourra être directement utilisé par TensorFlow, Keras ou PyTorch lors des phases d'entraînement et d'inférence.

In [17]:
with open(
    "../../01_data/metadata/label2id.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        label2id,
        f,
        indent=4,
        ensure_ascii=False
    )

print("label2id.json créé avec succès")


label2id.json créé avec succès


# Conclusion

Le Class Mapping a été généré avec succès.

Les 17 classes du dataset AgriLens disposent désormais d'un identifiant numérique unique et reproductible.

Les fichiers class_mapping.csv et label2id.json constituent désormais la référence officielle utilisée par les futures étapes Machine Learning, Deep Learning et MLOps du projet.